# Week 12: Forced Oscillations & Resonance — PHASE 5: Oscillations & Waves

*📚 Physics I (PHY101) · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Write** the equation of motion for a driven (forced) damped harmonic oscillator
2. **Derive** and interpret the steady-state amplitude and phase response
3. **Explain** the resonance phenomenon and identify the resonance frequency
4. **Calculate** the quality factor $Q$ and relate it to the sharpness of the resonance peak
5. **Analyze** amplitude-frequency and phase-frequency response curves
6. **Design** a simple vibration damper by selecting optimal damping parameters
7. **Simulate** driven oscillator dynamics using `scipy.integrate.odeint`

## 🎯 Core Mastery Connection

Drive an oscillator at its natural frequency and resonance occurs. This week you learn to predict resonance conditions and design dampers — skills critical for mechatronics and structural engineering. The workflow is the same: diagram the driven system, identify the driving and restoring forces, write the equation of motion, and predict the steady-state amplitude, phase, and power transfer.

---
## 1. Setup

Run this cell first to import all necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from scipy.integrate import odeint

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("All libraries loaded successfully!")

---
## 2. The Driven Damped Harmonic Oscillator

Last week we studied free oscillations (with and without damping). Now we add a **periodic driving force**:

$$m\ddot{x} + b\dot{x} + kx = F_0 \cos(\omega_d t)$$

Dividing by $m$ and defining $\gamma = b/(2m)$, $\omega_0 = \sqrt{k/m}$, $f_0 = F_0/m$:

$$\ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = f_0 \cos(\omega_d t)$$

### Analogy

Imagine pushing a child on a swing. You apply a periodic push (the driving force). If you push at the right moment (in sync with the natural swing frequency), the amplitude builds up dramatically. Push at the wrong frequency, and the swing barely responds. This is **resonance**.

### Steady-State Solution

After transients die out, the system settles into a steady-state oscillation at the **driving frequency** $\omega_d$:

$$x(t) = A(\omega_d)\cos(\omega_d t - \delta)$$

where the **steady-state amplitude** is:

$$A(\omega_d) = \frac{f_0}{\sqrt{(\omega_0^2 - \omega_d^2)^2 + (2\gamma\omega_d)^2}}$$

and the **phase lag** is:

$$\delta = \arctan\left(\frac{2\gamma\omega_d}{\omega_0^2 - \omega_d^2}\right)$$

### Key Features

| Feature | Description |
|---------|-------------|
| **Resonance frequency** | $\omega_r = \sqrt{\omega_0^2 - 2\gamma^2}$ (amplitude peaks here) |
| **At resonance** | Phase lag $\delta = 90°$ (response lags driving by quarter period) |
| **Low frequency** ($\omega_d \ll \omega_0$) | $A \approx f_0/\omega_0^2 = F_0/k$, $\delta \approx 0°$ |
| **High frequency** ($\omega_d \gg \omega_0$) | $A \to 0$, $\delta \to 180°$ |

---
## 3. Interactive Demo 1: Driven Oscillator Animation with Resonance Buildup

Watch how a driven oscillator builds up amplitude over time. The transient (natural frequency) eventually dies out, leaving only the steady-state response at the driving frequency.

In [ ]:
def driven_oscillator_animation(omega_d=None, gamma=0.15, omega0=2*np.pi, F0_over_m=5.0):
    """Animated driven oscillator showing resonance buildup."""
    if omega_d is None:
        omega_d = omega0  # Drive at resonance by default

    def driven_ode(state, t):
        x, v = state
        driving = F0_over_m * np.cos(omega_d * t)
        return [v, driving - 2 * gamma * v - omega0**2 * x]

    t_end = 30.0
    t_arr = np.linspace(0, t_end, 1500)
    sol = odeint(driven_ode, [0, 0], t_arr)
    x_arr = sol[:, 0]
    v_arr = sol[:, 1]

    # Steady-state amplitude
    A_ss = F0_over_m / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
    driving_arr = F0_over_m * np.cos(omega_d * t_arr) / omega0**2  # normalized

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
    plt.close(fig)

    # --- Top: Driving force ---
    ax1.set_xlim(0, t_end)
    force_max = F0_over_m / omega0**2 * 1.3
    ax1.set_ylim(-force_max, force_max)
    ax1.set_ylabel('Driving force / k', fontsize=11)
    ax1.set_title('Driving Force', fontweight='bold')
    drive_line, = ax1.plot([], [], 'orange', lw=2)

    # --- Middle: Response ---
    ax2.set_xlim(0, t_end)
    x_max = max(np.abs(x_arr).max() * 1.2, A_ss * 1.3)
    ax2.set_ylim(-x_max, x_max)
    ax2.set_ylabel('x (m)', fontsize=11)
    ax2.set_title(f'Response ($\\omega_d$={omega_d:.2f}, $\\omega_0$={omega0:.2f}, A_ss={A_ss:.2f} m)', fontweight='bold')
    ax2.axhline(y=A_ss, color='green', ls='--', lw=1.5, alpha=0.5, label=f'Steady-state A = {A_ss:.2f}')
    ax2.axhline(y=-A_ss, color='green', ls='--', lw=1.5, alpha=0.5)
    response_line, = ax2.plot([], [], 'b-', lw=2, label='x(t)')
    ax2.legend(loc='upper right', fontsize=9)

    # --- Bottom: Spring-mass visual ---
    ax3.set_xlim(-x_max * 1.5, x_max * 1.5)
    ax3.set_ylim(-0.8, 0.8)
    ax3.set_aspect('equal')
    ax3.set_title('Mass on Spring (driven)', fontweight='bold')
    ax3.set_xlabel('Position x (m)')
    ax3.set_yticks([])
    ax3.axvline(0, color='gray', ls='--', alpha=0.4)

    wall = plt.Rectangle((-x_max * 1.5, -0.5), 0.1, 1.0, color='gray')
    ax3.add_patch(wall)
    spring_line, = ax3.plot([], [], 'b-', lw=2)
    mass_patch = plt.Rectangle((0, -0.25), 0.5, 0.5, fc='royalblue', ec='navy', lw=2)
    ax3.add_patch(mass_patch)
    force_arrow, = ax3.plot([], [], 'r-', lw=3)
    info_text = ax3.text(x_max * 0.8, 0.6, '', fontsize=10, ha='center',
                          bbox=dict(boxstyle='round', fc='lightyellow'))

    def make_spring(x0, x1, n_coils=12, width=0.15):
        L = x1 - x0
        s = np.linspace(0, 1, n_coils * 20 + 1)
        sx = x0 + s * L
        sy = width * np.sin(2 * np.pi * n_coils * s)
        sy[0] = 0; sy[-1] = 0
        return sx, sy

    fig.tight_layout()

    step = 3  # frame stepping for speed

    def animate(frame):
        i = frame * step
        if i >= len(t_arr):
            i = len(t_arr) - 1

        drive_line.set_data(t_arr[:i+1], driving_arr[:i+1])
        response_line.set_data(t_arr[:i+1], x_arr[:i+1])

        x = x_arr[i]
        sx, sy = make_spring(-x_max * 1.5 + 0.1, x - 0.25)
        spring_line.set_data(sx, sy)
        mass_patch.set_xy((x - 0.25, -0.25))

        # Force arrow
        f_now = F0_over_m * np.cos(omega_d * t_arr[i])
        arrow_len = f_now / omega0**2 * 0.5
        force_arrow.set_data([x + 0.25, x + 0.25 + arrow_len], [0, 0])

        info_text.set_text(f't = {t_arr[i]:.1f} s\nx = {x:.3f} m')

        return drive_line, response_line, spring_line, mass_patch, force_arrow, info_text

    n_frames = len(t_arr) // step
    ani = animation.FuncAnimation(fig, animate, frames=n_frames,
                                  interval=25, blit=False)
    return HTML(ani.to_jshtml())

print("Driving AT resonance (omega_d = omega_0):")
display(driven_oscillator_animation(omega_d=2*np.pi, gamma=0.15, omega0=2*np.pi))

In [ ]:
print("Driving AWAY from resonance (omega_d = 1.5 * omega_0):")
display(driven_oscillator_animation(omega_d=3*np.pi, gamma=0.15, omega0=2*np.pi))

---
## 4. The Resonance Curve

The amplitude response function:

$$A(\omega_d) = \frac{f_0}{\sqrt{(\omega_0^2 - \omega_d^2)^2 + (2\gamma\omega_d)^2}}$$

This curve has a peak near $\omega_0$. The **width** and **height** of this peak are controlled by the damping $\gamma$:

- **Light damping** ($\gamma \ll \omega_0$): tall, narrow peak
- **Heavy damping** ($\gamma \sim \omega_0$): short, broad peak

The peak occurs at the **resonance frequency**:

$$\omega_r = \sqrt{\omega_0^2 - 2\gamma^2}$$

(For light damping, $\omega_r \approx \omega_0$.)

---
## 5. Interactive Demo 2: Amplitude vs Frequency Response Curve

Adjust the damping coefficient and observe how the resonance peak changes.

In [ ]:
def interactive_resonance_curve(gamma, omega0, f0):
    """Amplitude-frequency response with adjustable damping."""
    omega_d = np.linspace(0.01, 3 * omega0, 1000)

    A_response = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)

    # Phase
    delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)

    # Resonance frequency
    disc = omega0**2 - 2 * gamma**2
    if disc > 0:
        omega_r = np.sqrt(disc)
        A_max = f0 / (2 * gamma * np.sqrt(omega0**2 - gamma**2))
    else:
        omega_r = 0
        A_max = A_response[0]

    # Q factor
    Q = omega0 / (2 * gamma) if gamma > 0 else float('inf')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Amplitude response ---
    ax1.plot(omega_d / omega0, A_response, 'b-', lw=2.5)
    ax1.axvline(x=1.0, color='gray', ls='--', alpha=0.5, label=f'$\\omega_0$ = {omega0:.2f}')
    if disc > 0:
        ax1.axvline(x=omega_r / omega0, color='red', ls=':', lw=2,
                    label=f'$\\omega_r$ = {omega_r:.2f}')
        ax1.plot(omega_r / omega0, A_max, 'r*', ms=15, label=f'$A_{{max}}$ = {A_max:.2f}')

    # Half-power bandwidth
    A_half = A_max / np.sqrt(2) if disc > 0 else 0
    ax1.axhline(y=A_half, color='green', ls=':', alpha=0.5, label=f'$A_{{max}}/\\sqrt{{2}}$')

    ax1.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
    ax1.set_ylabel('Amplitude A (m)', fontsize=13)
    ax1.set_title(f'Amplitude Response\n$\\gamma$={gamma:.2f}, Q={Q:.1f}', fontweight='bold', fontsize=13)
    ax1.legend(fontsize=9)
    ax1.set_xlim(0, 3)

    # --- Phase response ---
    ax2.plot(omega_d / omega0, np.degrees(delta), 'r-', lw=2.5)
    ax2.axvline(x=1.0, color='gray', ls='--', alpha=0.5)
    ax2.axhline(y=90, color='green', ls=':', alpha=0.5, label='$\\delta = 90°$')
    ax2.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
    ax2.set_ylabel('Phase lag $\\delta$ (degrees)', fontsize=13)
    ax2.set_title('Phase Response', fontweight='bold', fontsize=13)
    ax2.set_ylim(-5, 185)
    ax2.set_xlim(0, 3)
    ax2.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

widgets.interact(interactive_resonance_curve,
    gamma=widgets.FloatSlider(value=0.3, min=0.05, max=5.0, step=0.05,
                              description='Damping $\\gamma$ (s$^{-1}$):',
                              style={'description_width': 'initial'}),
    omega0=widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5,
                               description='$\\omega_0$ (rad/s):',
                               style={'description_width': 'initial'}),
    f0=widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0,
                           description='$f_0 = F_0/m$ (m/s$^2$):',
                           style={'description_width': 'initial'})
);

### Multiple Damping Values on One Plot

In [ ]:
omega0 = 5.0
f0 = 10.0
gamma_values = [0.1, 0.3, 0.7, 1.5, 3.0]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(gamma_values)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

omega_d = np.linspace(0.01, 3 * omega0, 1000)

for gam, col in zip(gamma_values, colors):
    A_resp = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gam * omega_d)**2)
    delta = np.arctan2(2 * gam * omega_d, omega0**2 - omega_d**2)
    Q = omega0 / (2 * gam)

    ax1.plot(omega_d / omega0, A_resp, color=col, lw=2,
             label=f'$\\gamma$={gam:.1f} (Q={Q:.1f})')
    ax2.plot(omega_d / omega0, np.degrees(delta), color=col, lw=2,
             label=f'$\\gamma$={gam:.1f}')

ax1.axvline(x=1.0, color='gray', ls='--', alpha=0.4)
ax1.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
ax1.set_ylabel('Amplitude A (m)', fontsize=13)
ax1.set_title('Amplitude-Frequency Response', fontweight='bold', fontsize=14)
ax1.legend(fontsize=9)
ax1.set_xlim(0, 3)

ax2.axvline(x=1.0, color='gray', ls='--', alpha=0.4)
ax2.axhline(y=90, color='gray', ls=':', alpha=0.4)
ax2.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
ax2.set_ylabel('Phase lag $\\delta$ (degrees)', fontsize=13)
ax2.set_title('Phase-Frequency Response', fontweight='bold', fontsize=14)
ax2.legend(fontsize=9)
ax2.set_xlim(0, 3)
ax2.set_ylim(-5, 185)

plt.tight_layout()
plt.show()

---
## 6. Phase Diagram: Driving Frequency vs Response Phase

The phase relationship between the driving force and the response is crucial for understanding energy transfer:

- $\delta \approx 0°$: Force and displacement are in phase (stiffness-dominated)
- $\delta = 90°$: Force leads displacement by 90° (resonance, maximum power input)
- $\delta \approx 180°$: Force and displacement are out of phase (inertia-dominated)

---
## 7. Interactive Demo 3: Phase Diagram

Visualize how the response (blue) relates to the driving force (orange) at different frequencies.

In [ ]:
def interactive_phase_diagram(omega_d_ratio, gamma):
    """Show driving force vs response with phase relationship."""
    omega0 = 5.0
    f0 = 10.0
    omega_d = omega_d_ratio * omega0

    A_ss = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
    delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)

    t = np.linspace(0, 4 * 2 * np.pi / omega_d, 500)
    force = np.cos(omega_d * t)
    response = A_ss * np.cos(omega_d * t - delta)

    # Also solve ODE for transient
    def driven_ode(state, t):
        x, v = state
        return [v, f0 * np.cos(omega_d * t) - 2 * gamma * v - omega0**2 * x]

    t_long = np.linspace(0, 40, 3000)
    sol = odeint(driven_ode, [0, 0], t_long)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # --- Steady state comparison ---
    ax = axes[0, 0]
    ax.plot(t, force, 'orange', lw=2, label='Driving force (normalized)')
    ax.plot(t, response / A_ss, 'b-', lw=2, label='Response (normalized)')
    ax.set_xlabel('Time (s)')
    ax.set_title(f'Steady State: $\\delta$ = {np.degrees(delta):.1f}°', fontweight='bold')
    ax.legend(fontsize=9)

    # Phase arrow
    ax.annotate('', xy=(delta / omega_d, 0.9), xytext=(0, 0.9),
                arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax.text(delta / (2 * omega_d), 0.75, f'$\\delta$={np.degrees(delta):.0f}°',
            color='red', fontsize=11, ha='center')

    # --- Full transient ---
    ax = axes[0, 1]
    ax.plot(t_long, sol[:, 0], 'b-', lw=1.5)
    ax.axhline(y=A_ss, color='green', ls='--', alpha=0.5, label=f'$A_{{ss}}$={A_ss:.2f}')
    ax.axhline(y=-A_ss, color='green', ls='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('x (m)')
    ax.set_title('Full Response (transient + steady)', fontweight='bold')
    ax.legend(fontsize=9)

    # --- Phasor diagram ---
    ax = axes[1, 0]
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    circle = plt.Circle((0, 0), 1, fill=False, color='gray', ls='--', alpha=0.3)
    ax.add_patch(circle)

    # Force phasor
    ax.arrow(0, 0, 0.9 * np.cos(0), 0.9 * np.sin(0), head_width=0.08,
             fc='orange', ec='orange', lw=2)
    ax.text(1.1, 0.1, 'F (driving)', color='orange', fontsize=11, fontweight='bold')

    # Response phasor (lags by delta)
    ax.arrow(0, 0, 0.9 * np.cos(-delta), 0.9 * np.sin(-delta), head_width=0.08,
             fc='blue', ec='blue', lw=2)
    ax.text(0.9 * np.cos(-delta) + 0.1, 0.9 * np.sin(-delta) - 0.15,
            'x (response)', color='blue', fontsize=11, fontweight='bold')

    # Arc for phase angle
    arc_theta = np.linspace(-delta, 0, 50)
    ax.plot(0.5 * np.cos(arc_theta), 0.5 * np.sin(arc_theta), 'r-', lw=2)
    ax.text(0.55 * np.cos(-delta / 2), 0.55 * np.sin(-delta / 2) - 0.1,
            f'$\\delta$={np.degrees(delta):.0f}°', color='red', fontsize=12)

    ax.set_title('Phasor Diagram', fontweight='bold')

    # --- Power ---
    ax = axes[1, 1]
    power = f0 * np.cos(omega_d * t) * (-A_ss * omega_d * np.sin(omega_d * t - delta))
    ax.plot(t, power, 'm-', lw=2)
    ax.fill_between(t, 0, power, where=(power > 0), alpha=0.3, color='green', label='Energy in')
    ax.fill_between(t, 0, power, where=(power < 0), alpha=0.3, color='red', label='Energy out')
    P_avg = 0.5 * f0 * A_ss * omega_d * np.sin(delta)
    ax.axhline(y=P_avg, color='black', ls='--', lw=2, label=f'$\\langle P \\rangle$ = {P_avg:.2f}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Power (W/kg)')
    ax.set_title(f'Instantaneous Power', fontweight='bold')
    ax.legend(fontsize=9)

    fig.suptitle(f'$\\omega_d/\\omega_0$ = {omega_d_ratio:.2f}, $\\gamma$ = {gamma:.2f}, A = {A_ss:.3f} m',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(interactive_phase_diagram,
    omega_d_ratio=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05,
                                      description='$\\omega_d/\\omega_0$:',
                                      style={'description_width': 'initial'}),
    gamma=widgets.FloatSlider(value=0.3, min=0.05, max=3.0, step=0.05,
                              description='$\\gamma$ (s$^{-1}$):',
                              style={'description_width': 'initial'})
);

---
## 8. The Quality Factor (Q)

The **quality factor** $Q$ measures how "sharp" the resonance is. It is defined as:

$$Q = \frac{\omega_0}{2\gamma} = \frac{\omega_0}{\Delta\omega}$$

where $\Delta\omega$ is the **full width at half-maximum power** (FWHM) of the resonance curve.

| $Q$ value | Interpretation | Example |
|-----------|---------------|---------|
| $Q < 1$ | Overdamped, no resonance | Door closer |
| $Q \sim 1-10$ | Moderate damping | Car suspension |
| $Q \sim 100$ | Sharp resonance | Acoustic guitar |
| $Q \sim 10^4$ | Very sharp resonance | Quartz crystal oscillator |

Physical meaning: $Q$ is approximately $2\pi$ times the number of oscillation cycles before the energy drops to $1/e$ of its initial value.

$$Q \approx 2\pi \times \frac{\text{Energy stored}}{\text{Energy lost per cycle}}$$

---
## 9. Interactive Demo 4: Q Factor Visualizer

See how $Q$ controls the height and width of the resonance peak. The bandwidth (FWHM) is highlighted.

In [ ]:
def interactive_Q_factor(Q):
    """Visualize the Q factor and bandwidth."""
    omega0 = 10.0
    f0 = 10.0
    gamma = omega0 / (2 * Q)

    omega_d = np.linspace(0.01, 2.5 * omega0, 2000)
    A_resp = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)

    # Normalized response A / A_static where A_static = f0/omega0^2
    A_static = f0 / omega0**2
    A_norm = A_resp / A_static

    # Peak
    disc = omega0**2 - 2 * gamma**2
    if disc > 0:
        omega_r = np.sqrt(disc)
        A_peak_norm = A_norm[np.argmin(np.abs(omega_d - omega_r))]
    else:
        omega_r = 0
        A_peak_norm = A_norm[0]

    # Half-power level
    A_half = A_peak_norm / np.sqrt(2)

    # Find FWHM
    above_half = omega_d[A_norm >= A_half]
    if len(above_half) > 1:
        bw_low = above_half[0]
        bw_high = above_half[-1]
        bandwidth = bw_high - bw_low
    else:
        bw_low = bw_high = bandwidth = 0

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Resonance curve ---
    ax1.plot(omega_d / omega0, A_norm, 'b-', lw=2.5)
    ax1.axvline(x=1.0, color='gray', ls='--', alpha=0.4, label='$\\omega_0$')

    # Highlight bandwidth
    if bandwidth > 0:
        ax1.fill_between(omega_d / omega0, 0, A_norm,
                         where=(A_norm >= A_half), alpha=0.2, color='red')
        ax1.axhline(y=A_half, color='green', ls=':', lw=1.5)
        ax1.annotate('', xy=(bw_high / omega0, A_half * 0.8),
                     xytext=(bw_low / omega0, A_half * 0.8),
                     arrowprops=dict(arrowstyle='<->', color='red', lw=2))
        ax1.text((bw_low + bw_high) / (2 * omega0), A_half * 0.65,
                 f'$\\Delta\\omega$ = {bandwidth:.2f} rad/s',
                 color='red', fontsize=12, ha='center', fontweight='bold')

    ax1.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
    ax1.set_ylabel('$A / A_{static}$', fontsize=13)
    ax1.set_title(f'Resonance Curve\nQ = {Q:.1f}, $\\gamma$ = {gamma:.2f} s$^{{-1}}$',
                  fontweight='bold', fontsize=13)
    ax1.legend(fontsize=10)
    ax1.set_xlim(0, 2.5)

    # --- Energy decay in free oscillation ---
    t_free = np.linspace(0, 10 * 2 * np.pi / omega0, 1000)
    x_free = np.exp(-gamma * t_free) * np.cos(omega0 * t_free)
    E_free = np.exp(-2 * gamma * t_free)  # normalized

    n_cycles = t_free * omega0 / (2 * np.pi)

    ax2.plot(n_cycles, E_free, 'purple', lw=2.5, label='E(t) / E(0)')
    # Mark 1/e level
    ax2.axhline(y=1 / np.e, color='orange', ls='--', lw=2, label='$1/e$ level')
    ax2.axvline(x=Q / (2 * np.pi), color='orange', ls=':', alpha=0.5)
    ax2.set_xlabel('Number of cycles', fontsize=13)
    ax2.set_ylabel('E / E(0)', fontsize=13)
    ax2.set_title(f'Energy Decay (free oscillation)\n~{Q/(2*np.pi):.0f} cycles to 1/e',
                  fontweight='bold', fontsize=13)
    ax2.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"Summary:")
    print(f"  Q factor = {Q:.1f}")
    print(f"  gamma = {gamma:.3f} s^-1")
    print(f"  Peak amplification = {A_peak_norm:.1f}x static")
    print(f"  Bandwidth Delta_omega = {bandwidth:.2f} rad/s")
    print(f"  omega_0 / Delta_omega = {omega0/bandwidth:.1f}" if bandwidth > 0 else "")

widgets.interact(interactive_Q_factor,
    Q=widgets.FloatSlider(value=5.0, min=0.5, max=50.0, step=0.5,
                          description='Q factor:',
                          style={'description_width': 'initial'})
);

---
## 10. Engineering Application: Vibration Dampers

In engineering, vibration dampers (also called tuned mass dampers) protect structures from resonance. The Taipei 101 skyscraper has a 730-ton steel pendulum that swings to counteract wind-induced oscillations.

### The Design Problem

Given a structure with natural frequency $\omega_0$ that is subject to a periodic driving force (e.g., wind, machinery, earthquakes), you need to choose a damping coefficient $b$ to keep the vibration amplitude below a safe limit.

**Trade-offs:**
- Too little damping: large resonance amplitude (dangerous!)
- Too much damping: system responds slowly, poor vibration isolation at high frequencies
- Optimal: balance between peak reduction and broadband isolation

---
## 11. Interactive Demo 5: Vibration Damper Design Tool

Design a vibration damper for a machine base. Adjust parameters and see if the maximum vibration stays below the safety limit.

In [ ]:
def vibration_damper_tool(m, k, b, F0, rpm_min, rpm_max, x_safe):
    """Interactive vibration damper design tool."""
    omega0 = np.sqrt(k / m)
    gamma = b / (2 * m)
    f0 = F0 / m
    Q = omega0 / (2 * gamma) if gamma > 0 else float('inf')

    # Convert RPM range to angular frequency
    omega_min = rpm_min * 2 * np.pi / 60
    omega_max = rpm_max * 2 * np.pi / 60
    omega_d = np.linspace(omega_min, omega_max, 1000)

    A_resp = (F0 / k) / np.sqrt((1 - (omega_d / omega0)**2)**2 + (2 * gamma * omega_d / omega0**2)**2)

    # Full range for reference
    omega_full = np.linspace(0.01, omega_max * 1.5, 1000)
    A_full = (F0 / k) / np.sqrt((1 - (omega_full / omega0)**2)**2 + (2 * gamma * omega_full / omega0**2)**2)
    rpm_full = omega_full * 60 / (2 * np.pi)

    rpm_range = omega_d * 60 / (2 * np.pi)
    A_max = np.max(A_resp)
    safe = A_max <= x_safe

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Amplitude vs RPM ---
    ax1.plot(rpm_full, A_full * 1000, 'b-', lw=2, label='Response')
    ax1.axhspan(0, x_safe * 1000, alpha=0.1, color='green', label='Safe zone')
    ax1.axhline(y=x_safe * 1000, color='green', ls='--', lw=2)
    ax1.axvspan(rpm_min, rpm_max, alpha=0.15, color='orange', label='Operating range')
    ax1.axhline(y=A_max * 1000, color='red' if not safe else 'green', ls=':', lw=2,
                label=f'Max in range: {A_max*1000:.2f} mm')

    f0_hz = omega0 / (2 * np.pi)
    rpm_resonance = f0_hz * 60
    ax1.axvline(x=rpm_resonance, color='gray', ls='--', alpha=0.5)
    ax1.text(rpm_resonance, ax1.get_ylim()[1] * 0.95,
             f'f$_0$={f0_hz:.1f} Hz\n{rpm_resonance:.0f} RPM',
             ha='center', fontsize=9, color='gray')

    ax1.set_xlabel('Frequency (RPM)', fontsize=12)
    ax1.set_ylabel('Amplitude (mm)', fontsize=12)
    status = 'SAFE' if safe else 'EXCEEDS LIMIT'
    status_color = 'green' if safe else 'red'
    ax1.set_title(f'Vibration Response — {status}', fontweight='bold',
                  fontsize=14, color=status_color)
    ax1.legend(fontsize=9, loc='upper right')

    # --- Parameter summary ---
    ax2.axis('off')
    summary = [
        ['Parameter', 'Value'],
        ['Mass m', f'{m:.1f} kg'],
        ['Stiffness k', f'{k:.0f} N/m'],
        ['Damping b', f'{b:.1f} N s/m'],
        ['Natural freq $\omega_0$', f'{omega0:.2f} rad/s ({omega0/(2*np.pi):.2f} Hz)'],
        ['Damping ratio $\gamma$', f'{gamma:.3f} s$^{{-1}}$'],
        ['Q factor', f'{Q:.1f}'],
        ['Driving force $F_0$', f'{F0:.1f} N'],
        ['Operating range', f'{rpm_min:.0f} - {rpm_max:.0f} RPM'],
        ['Max amplitude in range', f'{A_max*1000:.2f} mm'],
        ['Safety limit', f'{x_safe*1000:.1f} mm'],
        ['Status', status],
    ]

    table = ax2.table(cellText=[[r[0], r[1]] for r in summary],
                      colWidths=[0.45, 0.45], loc='center', cellLoc='left')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.8)

    # Color the status row
    table[len(summary) - 1, 1].set_facecolor('lightgreen' if safe else 'lightcoral')
    table[0, 0].set_facecolor('lightblue')
    table[0, 1].set_facecolor('lightblue')

    ax2.set_title('Design Parameters', fontweight='bold', fontsize=14)

    plt.tight_layout()
    plt.show()

widgets.interact(vibration_damper_tool,
    m=widgets.FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0,
                          description='Mass m (kg):', style={'description_width': 'initial'}),
    k=widgets.FloatSlider(value=5000.0, min=500.0, max=50000.0, step=500.0,
                          description='Stiffness k (N/m):', style={'description_width': 'initial'}),
    b=widgets.FloatSlider(value=100.0, min=1.0, max=2000.0, step=10.0,
                          description='Damping b (N s/m):', style={'description_width': 'initial'}),
    F0=widgets.FloatSlider(value=50.0, min=5.0, max=500.0, step=5.0,
                           description='Force F0 (N):', style={'description_width': 'initial'}),
    rpm_min=widgets.FloatSlider(value=300, min=0, max=3000, step=50,
                                description='RPM min:', style={'description_width': 'initial'}),
    rpm_max=widgets.FloatSlider(value=1800, min=100, max=5000, step=50,
                                description='RPM max:', style={'description_width': 'initial'}),
    x_safe=widgets.FloatSlider(value=0.005, min=0.001, max=0.05, step=0.001,
                               description='Safety limit (m):', style={'description_width': 'initial'},
                               readout_format='.3f')
);

---
## 12. Worked Examples

### Example 1: Resonance of a Car on a Bumpy Road

A car (1500 kg, suspension $k = 60000$ N/m, $b = 3000$ N$\cdot$s/m) drives over speed bumps spaced 10 m apart. At what speed does the car experience resonance? What is the Q factor?

In [ ]:
m_car = 1500     # kg
k_car = 60000    # N/m
b_car = 3000     # N·s/m
bump_spacing = 10  # m

omega0 = np.sqrt(k_car / m_car)
gamma = b_car / (2 * m_car)
f0_hz = omega0 / (2 * np.pi)
Q = omega0 / (2 * gamma)

# Resonance speed: v = f0 * spacing
v_resonance = f0_hz * bump_spacing
v_resonance_kmh = v_resonance * 3.6

print("=" * 55)
print("Example 1: Car on Bumpy Road")
print("=" * 55)
print(f"Natural frequency: omega_0 = {omega0:.2f} rad/s")
print(f"                  f_0 = {f0_hz:.2f} Hz")
print(f"Damping: gamma = {gamma:.2f} s^-1")
print(f"Q factor: {Q:.2f}")
print(f"\nResonance speed: v = f_0 * d = {f0_hz:.2f} * {bump_spacing}")
print(f"                 v = {v_resonance:.1f} m/s = {v_resonance_kmh:.0f} km/h")
print(f"\nAt this speed, the car bounces with maximum amplitude!")

### Example 2: Steady-State Amplitude

A 2 kg mass on a spring ($k = 200$ N/m) with damping $b = 4$ N$\cdot$s/m is driven by $F(t) = 10\cos(8t)$ N. Find the steady-state amplitude and phase lag.

In [ ]:
m = 2.0; k = 200.0; b = 4.0; F0 = 10.0; omega_d = 8.0

omega0 = np.sqrt(k / m)
gamma = b / (2 * m)
f0 = F0 / m

A_ss = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)
A_static = F0 / k

print("=" * 55)
print("Example 2: Steady-State Amplitude")
print("=" * 55)
print(f"omega_0 = sqrt({k}/{m}) = {omega0:.2f} rad/s")
print(f"gamma = {b}/(2*{m}) = {gamma:.2f} s^-1")
print(f"omega_d = {omega_d:.2f} rad/s")
print(f"\nSteady-state amplitude: A = {A_ss:.4f} m = {A_ss*1000:.2f} mm")
print(f"Phase lag: delta = {np.degrees(delta):.1f} degrees")
print(f"Static deflection: x_static = F0/k = {A_static:.4f} m")
print(f"Amplification factor: A/x_static = {A_ss/A_static:.2f}")

# Verify with odeint
def driven_ode(state, t):
    return [state[1], f0 * np.cos(omega_d * t) - 2 * gamma * state[1] - omega0**2 * state[0]]

t = np.linspace(0, 30, 5000)
sol = odeint(driven_ode, [0, 0], t)
A_numerical = np.max(np.abs(sol[-2000:, 0]))
print(f"\nNumerical verification (last portion): A_max = {A_numerical:.4f} m")

### Example 3: Q Factor and Bandwidth

An RLC circuit has a resonant frequency of 1000 Hz and a bandwidth of 20 Hz. Find the Q factor. If the resistance is doubled, what is the new bandwidth?

---
## Problem Set

**Instructions:** Solve each problem analytically first, then verify your answer numerically in the code cell below it. Show your work with clear variable definitions and unit tracking.

- **L1 (Basic):** Straightforward single-concept problems
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

### L1 (Basic) — P1

A damped oscillator has natural frequency $\omega_0 = 10$ rad/s and damping $\gamma = 0.5$ s$^{-1}$. Calculate the quality factor $Q$.

<details><summary>Answer</summary>$Q = 10$</details>

In [ ]:
# ✏️ [P1] Your solution here

### L1 (Basic) — P2

A $3.0$ kg mass on a spring ($k = 300$ N/m) is driven at the natural frequency with a force amplitude $F_0 = 6.0$ N. The damping coefficient is $b = 3.0$ N$\cdot$s/m. Find the steady-state amplitude at resonance.

<details><summary>Answer</summary>$A = 0.200$ m</details>

In [ ]:
# ✏️ [P2] Your solution here

### L1 (Basic) — P3

A forced oscillator has $\omega_0 = 20$ rad/s. It is driven at $\omega_d = 15$ rad/s with $\gamma = 2.0$ s$^{-1}$. Calculate the phase lag $\delta$ between the driving force and the response.

<details><summary>Answer</summary>$\delta = 19.3^\circ$</details>

In [ ]:
# ✏️ [P3] Your solution here

### L1 (Basic) — P4

A driven harmonic oscillator reaches steady state with amplitude $A = 0.050$ m at driving frequency $\omega_d = 25$ rad/s. The damping is $\gamma = 1.5$ s$^{-1}$. What is the average power delivered to the oscillator? (Use $\langle P \rangle = m\gamma\omega_d^2 A^2$.  Take $m = 2.0$ kg.)

<details><summary>Answer</summary>$\langle P \rangle = 4.69$ W</details>

In [ ]:
# ✏️ [P4] Your solution here

### L2 (Intermediate) — P5

A machine base ($m = 80$ kg) is mounted on springs with total stiffness $k = 32{,}000$ N/m and total damping $b = 640$ N$\cdot$s/m. The machine generates a sinusoidal force $F_0 = 200$ N at $1200$ RPM. Find (a) the natural frequency in Hz, (b) the quality factor, (c) the steady-state vibration amplitude, and (d) whether the amplitude exceeds a safety limit of $0.5$ mm.

<details><summary>Answer</summary>(a) $f_0 = \omega_0/2\pi = 20/2\pi = 3.18$ Hz; (b) $\gamma = b/2m = 4.0$ s$^{-1}$, so $Q = \omega_0/2\gamma = 2.5$; (c) $A = \dfrac{F_0/m}{\sqrt{(\omega_0^2-\omega_d^2)^2 + (2\gamma\omega_d)^2}} = \dfrac{2.5}{15424} = 0.162$ mm; (d) No — 0.162 mm is well within the 0.5 mm limit. **[CORRECTED]** previously $Q = 5.0$ and $A = 0.040$ mm; the conclusion in (d) is unchanged.</details>

In [ ]:
# ✏️ [P5] Your solution here

### L2 (Intermediate) — P6

A car ($1400$ kg, suspension $k = 55{,}000$ N/m, $b = 4000$ N$\cdot$s/m) drives over sinusoidal speed bumps spaced $8.0$ m apart. (a) At what speed does the car experience maximum bouncing (resonance)? (b) What is the Q factor of the suspension? (c) Is the system underdamped or overdamped?

<details><summary>Answer</summary>(a) $\omega_0 = \sqrt{55000/1400} = 6.268$ rad/s, $f_0 = 0.998$ Hz, so $v = f_0\lambda = 7.98$ m/s $= 28.7$ km/h; (b) $\gamma = b/2M = 1.4286$ s$^{-1}$, so $Q = \omega_0/2\gamma = 2.194$ (equivalently $\sqrt{kM}/b$); (c) Underdamped. **[CORRECTED]** (b) previously 2.46.</details>

In [ ]:
# ✏️ [P6] Your solution here

### L2 (Intermediate) — P7

An RLC circuit has $L = 0.10$ H, $C = 10$ $\mu$F, and $R = 20$ $\Omega$. (a) Find the resonant frequency $f_0$. (b) Find the quality factor. (c) What is the bandwidth $\Delta f$? (d) If $R$ is reduced to $5$ $\Omega$, what happens to $Q$ and the bandwidth?

<details><summary>Answer</summary>(a) $f_0 = 159.2$ Hz; (b) $Q = 5.0$; (c) $\Delta f = 31.8$ Hz; (d) $Q = 20$, $\Delta f = 7.96$ Hz</details>

In [ ]:
# ✏️ [P7] Your solution here

### L2 (Intermediate) — P8

A $0.50$ kg mass on a spring ($k = 50$ N/m) with $b = 1.0$ N$\cdot$s/m is driven by $F(t) = 3.0\cos(\omega_d t)$ N. Find the steady-state amplitude for (a) $\omega_d = 5$ rad/s, (b) $\omega_d = 10$ rad/s (at resonance), and (c) $\omega_d = 15$ rad/s.

<details><summary>Answer</summary>With $\omega_0 = 10$ rad/s and $\gamma = 1.0$ s$^{-1}$, $A = \dfrac{F_0/m}{\sqrt{(\omega_0^2-\omega_d^2)^2 + (2\gamma\omega_d)^2}}$: (a) $A = 6/\sqrt{5725} = 0.0793$ m; (b) $A = 6/20 = 0.300$ m; (c) $A = 6/\sqrt{16525} = 0.0467$ m. **[CORRECTED]** (a) and (c) previously 0.0800 and 0.0480 m — those drop the $(2\gamma\omega_d)^2$ term, which is inconsistent with using the full formula in (b). Note also that the amplitude peak is at $\sqrt{\omega_0^2-2\gamma^2} = 9.899$ rad/s, not exactly at $\omega_0$.</details>

In [ ]:
# ✏️ [P8] Your solution here

### L3 (Challenge) — P9

A sensitive optical table ($m = 500$ kg) must be isolated from floor vibrations in the range $5$--$50$ Hz. The table is mounted on pneumatic isolators modeled as springs with adjustable damping. (a) What spring constant $k$ is needed so that the natural frequency is $2.0$ Hz (below the vibration range)? (b) What damping ratio $\zeta = \gamma/\omega_0$ should be chosen so the transmissibility $T = A_\text{table}/A_\text{floor} < 0.05$ at $5$ Hz? (c) What is $T$ at $50$ Hz with this damping?

<details><summary>Answer</summary>(a) $k = m(2\pi f_0)^2 = 500(4\pi)^2 = 7.896\times10^4$ N/m. (b) **No damping ratio satisfies this requirement.** At 5 Hz, $r = 2.5$ and $T = \sqrt{\dfrac{1+25\zeta^2}{27.5625+25\zeta^2}}$, which *increases* with $\zeta$; its smallest possible value is $T(\zeta = 0) = 1/\sqrt{27.5625} = 0.1905$, already 3.81$\times$ above the 0.05 target. Reaching $T < 0.05$ at 5 Hz needs $r > 4.583$, i.e. $f_0 < 1.09$ Hz, not 2.0 Hz. (c) Answerable only once (b) is restated. Under the substitute criterion $T(f_0) \le 5$, $\zeta = 1/\sqrt{96} = 0.1021$ and $T(50\,\text{Hz}) = 0.0083$ ($-41.6$ dB). **[CORRECTED]** part (b) as written is infeasible; the problem needs a reachable target (or a softer mount).</details>

In [ ]:
# ✏️ [P9] Your solution here

### L3 (Challenge) — P10

A tuned mass damper (TMD) is to be designed for a bridge deck that resonates dangerously at $f_0 = 1.8$ Hz under pedestrian loading. The bridge effective mass is $M = 50{,}000$ kg, stiffness $K = 6.40 \times 10^6$ N/m, and structural damping $B = 5000$ N$\cdot$s/m. The TMD consists of a secondary mass $m_d = 2500$ kg (5% of bridge mass) on a spring $k_d$ with damper $b_d$. (a) What spring constant $k_d$ should be used so the TMD natural frequency matches the bridge? (b) For optimal damping, use $\zeta_d = \sqrt{3\mu/8}$ where $\mu = m_d/M$. Find $b_d$. (c) Estimate the reduction factor in peak response amplitude.

<details><summary>Answer</summary>(a) $k_d = m_d\omega_0^2 = 2500(2\pi\cdot1.8)^2 = 3.20\times10^5$ N/m. (b) $\mu = 0.05$, $\zeta_d = \sqrt{3\mu/8} = 0.13693$, so $b_d = 2\zeta_d m_d\omega_d = 2(0.13693)(2500)(11.3097) = 7743$ N$\cdot$s/m. (c) Peak response is reduced by roughly 70%. **[CORRECTED]** (b) previously 2732 N$\cdot$s/m, which corresponds to $\zeta = 0.0483$ rather than 0.1369.</details>

In [ ]:
# ✏️ [P10] Your solution here

---
## 13. Bridge to Next Week

This week we explored the rich physics of **forced oscillations and resonance**. We learned that:

- A periodic driving force produces a steady-state response at the driving frequency
- The amplitude peaks at resonance, controlled by the quality factor $Q$
- The phase lag transitions from 0 to 180 degrees as you sweep through resonance
- Engineers must carefully manage resonance in real structures and machines

These concepts extend far beyond mechanics. Resonance appears in:

- **Electrical circuits** (RLC resonance, radio tuning)
- **Optics** (Fabry-Perot cavities, laser resonance)
- **Acoustics** (musical instruments, room modes)
- **Quantum mechanics** (atomic transitions, NMR)

**Next week** we move to a different but equally fundamental topic: **Mechanical Waves**. You will see how oscillations at one point can propagate through a medium, creating traveling and standing waves. The mathematics of SHM you have mastered will be essential for understanding wave phenomena.